In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import requests

In [37]:
# Saving the URL for the API request and the header to retrieve csv data
url = 'https://esploradati.istat.it/SDMXWS/rest/data/41_983'
header = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}

# make the HTTP request
r = requests.get(url, headers=header)

# Check the status code
r.status_code


200

In [46]:
# converting the file into a text
df_text = r.text
print(df_text[:100])

DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_


In [58]:
# Saving the file into a csv format
with open('df_csv', 'w', encoding='utf-8') as f:
    f.write(df_text)

# Coverting the csv file into a pandas dataframe
df = pd.read_csv('df_csv')
df.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Checking for NaN values
df.isna().sum()

DATAFLOW                 0
FREQ                     0
REF_AREA                 0
DATA_TYPE                0
RESULT                   0
TIME_PERIOD              0
OBS_VALUE                0
OBS_STATUS          573552
NOTE_DS             573552
NOTE_REF_AREA       573552
NOTE_DATA_TYPE      573552
NOTE_RESULT         573552
NOTE_TIME_PERIOD    573552
BASE_PER            573552
UNIT_MEAS           573552
UNIT_MULT           573552
dtype: int64

In [57]:
for col, rig in df.items():
    print(col, rig.unique())

DATAFLOW <StringArray>
['IT1:41_983(1.0)']
Length: 1, dtype: str
FREQ <StringArray>
['A']
Length: 1, dtype: str
REF_AREA [  1001   1002   1003 ... 111105 111106 111107]
DATA_TYPE <StringArray>
['KILLINJ', 'ROADACC']
Length: 2, dtype: str
RESULT <StringArray>
['F', 'M', '9']
Length: 3, dtype: str
TIME_PERIOD [2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014
 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
OBS_VALUE [  10    7   13 ... 1081  880  969]
OBS_STATUS [nan]
NOTE_DS [nan]
NOTE_REF_AREA [nan]
NOTE_DATA_TYPE [nan]
NOTE_RESULT [nan]
NOTE_TIME_PERIOD [nan]
BASE_PER [nan]
UNIT_MEAS [nan]
UNIT_MULT [nan]


In [ ]:
# Creating a new dataframe by removing NaN columns and columns with the same data on it
df_filtered = df[['REF_AREA', 'DATA_TYPE', 'RESULT', 'TIME_PERIOD', 'OBS_VALUE']]

In [64]:
df_filtered.head()

,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE
0,1001,KILLINJ,F,2001,10
1,1001,KILLINJ,F,2002,10
2,1001,KILLINJ,F,2003,7
3,1001,KILLINJ,F,2004,13
4,1001,KILLINJ,F,2005,2


In [65]:
# Web scraping the data to get info about area population and cities
from selenium import webdriver
from selenium.webdriver.common.by import By

In [72]:
chrome_options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options = chrome_options)
driver.get('https://situas.istat.it/web/#/territorio/body?id=74&dateFrom=2020-12-31')

In [95]:
columns = driver.find_elements(By.CLASS_NAME, 'mr-1')
column_names = [name.text for name in columns]
column_names

['Codice Ripartizione geografica',
 'Codice Regione',
 'Codice Provincia (Storico)',
 'Codice Provincia/Uts',
 'Codice Comune (alfanumerico)',
 'Codice Comune (numerico)',
 'Comune',
 'Comune (dizione straniera)',
 'Sigla automobilistica',
 'Capoluogo di Provincia/Uts',
 'Capoluogo di Regione',
 'Popolazione legale',
 'Anno Censimento',
 'Superficie (Kmq)',
 'Anno (Superficie)',
 'Popolazione residente',
 'Anno (Popolazione residente)']

In [ ]:
# Removing the column since it is empty
column_names.remove('Comune (dizione straniera)')

In [116]:
df_comuni = pd.DataFrame(columns=column_names)
df_comuni

,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)


In [104]:
rows = driver.find_elements(By.CSS_SELECTOR, 'tbody.ant-table-tbody tr.ant-table-row')
rows_el = [el.text.strip() for el in rows]
rows_el

['1\n01\n001\n201\n001001\n1001\nAgliè\nTO\n0\n0\n2644\n2011\n13,1462\n2020\n2545\n2020',
 '1\n01\n001\n201\n001002\n1002\nAirasca\nTO\n0\n0\n3819\n2011\n15,7393\n2020\n3633\n2020',
 '1\n01\n001\n201\n001003\n1003\nAla di Stura\nTO\n0\n0\n462\n2011\n46,3315\n2020\n459\n2020',
 "1\n01\n001\n201\n001004\n1004\nAlbiano d'Ivrea\nTO\n0\n0\n1791\n2011\n11,7314\n2020\n1638\n2020",
 '1\n01\n001\n201\n001006\n1006\nAlmese\nTO\n0\n0\n6303\n2011\n17,8756\n2020\n6355\n2020',
 '1\n01\n001\n201\n001007\n1007\nAlpette\nTO\n0\n0\n277\n2011\n5,6261\n2020\n235\n2020',
 '1\n01\n001\n201\n001008\n1008\nAlpignano\nTO\n0\n0\n16893\n2011\n11,9193\n2020\n16484\n2020',
 '1\n01\n001\n201\n001009\n1009\nAndezeno\nTO\n0\n0\n1966\n2011\n7,486\n2020\n2013\n2020',
 '1\n01\n001\n201\n001010\n1010\nAndrate\nTO\n0\n0\n512\n2011\n9,3085\n2020\n488\n2020',
 '1\n01\n001\n201\n001011\n1011\nAngrogna\nTO\n0\n0\n870\n2011\n38,8782\n2020\n838\n2020']

In [117]:
rows_split = [r.split("\n") for r in rows_el]

In [118]:
rows_split

[['1',
  '01',
  '001',
  '201',
  '001001',
  '1001',
  'Agliè',
  'TO',
  '0',
  '0',
  '2644',
  '2011',
  '13,1462',
  '2020',
  '2545',
  '2020'],
 ['1',
  '01',
  '001',
  '201',
  '001002',
  '1002',
  'Airasca',
  'TO',
  '0',
  '0',
  '3819',
  '2011',
  '15,7393',
  '2020',
  '3633',
  '2020'],
 ['1',
  '01',
  '001',
  '201',
  '001003',
  '1003',
  'Ala di Stura',
  'TO',
  '0',
  '0',
  '462',
  '2011',
  '46,3315',
  '2020',
  '459',
  '2020'],
 ['1',
  '01',
  '001',
  '201',
  '001004',
  '1004',
  "Albiano d'Ivrea",
  'TO',
  '0',
  '0',
  '1791',
  '2011',
  '11,7314',
  '2020',
  '1638',
  '2020'],
 ['1',
  '01',
  '001',
  '201',
  '001006',
  '1006',
  'Almese',
  'TO',
  '0',
  '0',
  '6303',
  '2011',
  '17,8756',
  '2020',
  '6355',
  '2020'],
 ['1',
  '01',
  '001',
  '201',
  '001007',
  '1007',
  'Alpette',
  'TO',
  '0',
  '0',
  '277',
  '2011',
  '5,6261',
  '2020',
  '235',
  '2020'],
 ['1',
  '01',
  '001',
  '201',
  '001008',
  '1008',
  'Alpignano',
 

In [119]:
df_comuni = pd.DataFrame(rows_split, columns=column_names)

In [120]:
df_comuni

,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)
0,1,01,001,201,001001,1001,Agliè,TO,0,0,2644,2011,"13,1462",2020,2545,2020
1,1,01,001,201,001002,1002,Airasca,TO,0,0,3819,2011,"15,7393",2020,3633,2020
2,1,01,001,201,001003,1003,Ala di Stura,TO,0,0,462,2011,"46,3315",2020,459,2020
3,1,01,001,201,001004,1004,Albiano d'Ivrea,TO,0,0,1791,2011,"11,7314",2020,1638,2020
4,1,01,001,201,001006,1006,Almese,TO,0,0,6303,2011,"17,8756",2020,6355,2020
5,1,01,001,201,001007,1007,Alpette,TO,0,0,277,2011,"5,6261",2020,235,2020
6,1,01,001,201,001008,1008,Alpignano,TO,0,0,16893,2011,"11,9193",2020,16484,2020
7,1,01,001,201,001009,1009,Andezeno,TO,0,0,1966,2011,"7,486",2020,2013,2020
8,1,01,001,201,001010,1010,Andrate,TO,0,0,512,2011,"9,3085",2020,488,2020
9,1,01,001,201,001011,1011,Angrogna,TO,0,0,870,2011,"38,8782",2020,838,2020


In [ ]:
# Function for column list creation
def create_col(class_name):
    s_column = driver.find_elements(By.CLASS_NAME, class_name)
    column = [col.text for col in s_column]
    return column

# Function for page scraping
def scrape_page(page):
    driver.find_element(By.LINK_TEXT, str(page)).click()

    

    # Create a temporary dataframe to store the current page
    df_tmp = pd.DataFrame(
        {'Codice Comune (numerico)': Codice Comune (numerico),
         'Comune': Comune,
         'Capoluogo di Provincia/Uts': Capoluogo di Provincia/Uts,
         'Capoluogo di Regione': Capoluogo di Regione,
         'Popolazione legale': Popolazione legale,
         'Anno Censimento': Anno Censimento,
         'Superficie (Kmq)': Superficie (Kmq),
         'Anno (Superficie)': Anno (Superficie),
         'Popolazione residente': Popolazione residente,
         'Anno (Popolazione residente)': Anno (Popolazione residente)
         })
    
    return df_tmp

In [ ]:
# Creating a list of pages
driver.find_elements(By.LINK_TEXT, )